This is an example of generating MODIS-LAI h5 files to drive ATS 2D transect simulations.

- modified based on `get_MODIS-LAI.ipynb` in watershed-workflow
- Input
    - `data-processed/{site_name}/m2_{site_name}_nx100.mat`
    - `notebooks/MODIS_raw` data downloaded from APPEEARS website
- Output
    - `data-processed/{watershed_name}/{watershed_name}_MODIS_LAI_20021001_20250125.h5`: raw MODIS
    - ~~`data-processed/{watershed_name}/{watershed_name}_MODIS_LAI_2013-1-1_2024-1-1_smoothed.h5`: smoothed MODIS for watershed-scale ATS simulations~~
    - ~~`data-processed/{watershed_name}/{watershed_name}_MODIS_LAI_typical10yr_2013_2023.h5`: typical year MODIS for watershed-scale ATS spinup simulations~~
    - `data-processed/{site_name}/{site_name}_MODIS_LAI_20021001_20250125.h5`: raw MODIS for hillslope
    - `data-processed/{site_name}/{site_name}_MODIS_LAI_2013-1-1_2024-1-1_smoothed.h5`: smoothed MODIS for specified start_year to end_year
    - `data-processed/{site_name}/{site_name}_MODIS_LAI_typical10yr_2013_2023.h5`: typical year MODIS for spinup

**File History**

update 2025/8/32
- add `config.json`

In [ ]:
%load_ext autoreload
%autoreload 2

# Parameters and data sources

In [ ]:
# Parameters cell
import json
with open('config.json', 'r') as f:
    config = json.load(f)
watershed_name = config['watershed_name']
# hucs           = [config['hucs']]
site_name      = config['site_name']

# simulation control
start_year     = config['start_year']
end_year       = config['end_year']
nyears_cyclic_steadystate = config['nyears_cyclic_steadystate']

In [ ]:
outputs={}

In [ ]:
import os, sys
import xarray as xr # rioxarray required
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py as h5
import geopandas as gpd
from shapely.geometry import mapping
import netCDF4 as nc
from datetime import date, datetime, timedelta
import calendar
import shutil

import geopandas as gpd
from scipy.io import loadmat
import shapely
from shapely.geometry import Point, LineString, Polygon, box, mapping

import watershed_workflow
import watershed_workflow.source_list
import watershed_workflow.ui
import watershed_workflow.colors
import watershed_workflow.condition
import watershed_workflow.mesh
import watershed_workflow.split_hucs
import watershed_workflow.soil_properties
import watershed_workflow.daymet
import watershed_workflow.utils
import watershed_workflow.regions
import watershed_workflow.land_cover_properties

import scipy
import pyproj

plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = (9,5)
pd.options.display.max_columns = None
pd.options.display.max_rows = 20

In [ ]:
# Note that, by default, we tend to work in the DayMet CRS because this allows us to avoid
# reprojecting meteorological forcing datasets.
crs_daymet = watershed_workflow.crs.daymet_crs()
crs_latlon = watershed_workflow.crs.latlon_crs() # essentially epsg(4269)
# note: epsg(4269) i.e. NAD83 vs epsg(4326) i.e. WGS84
# - https://gis.stackexchange.com/questions/170839/is-re-projection-needed-from-srid-4326-wgs-84-to-srid-4269-nad-83

# alternative
#proj_daymet = "+proj=lcc +lat_1=25 +lat_2=60 +lat_0=42.5 +lon_0=-100 +x_0=0 +y_0=0 +datum=WGS84" # daymet crs
#proj_wgs84  = "epsg:4326" # latlon
#crs_daymet  = watershed_workflow.crs.from_string(proj_daymet)
#crs_wgs84   = watershed_workflow.crs.from_string(proj_wgs84)

# prepare watershed shape and hillslope shape

In [ ]:
# load from watershed shp
#watershed_name = 'OakCreek' # name the domain, used in filenames, etc
fname_watershed_shp = f'../data-processed/{watershed_name}/{watershed_name}_bounds.shp'
watershed_shape = gpd.read_file(fname_watershed_shp)

In [ ]:
# ## Import shapefile to provide the coordinates - not used anymore?
# def get_bounds(fname_watershed_shp):
#     """get the min,max bounds for lat,lon for a given watershed."""
#     watershed_shape = gpd.read_file(fname_watershed_shp)

#     bounds = watershed_shape.bounds
#     # bounds

#     lonl = watershed_shape.bounds['minx'].values[0]
#     lonr = watershed_shape.bounds['maxx'].values[0]
#     latb = watershed_shape.bounds['miny'].values[0]
#     latt = watershed_shape.bounds['maxy'].values[0]    
    
#     return lonl,lonr,latb,latt

# lonl,lonr,latb,latt = get_bounds(fname_watershed_shp)

# lonl,lonr,latb,latt

In [ ]:
# load hillslope geometry from mat file generated in "1-full_workflow_OakCreek.ipynb"
#site_name = 'NF01'
meshsize_nx = 100

m2_mat_filename =  f'../data-processed/{site_name}/m2_{site_name}_nx{meshsize_nx}.mat'
loaded_data = loadmat(m2_mat_filename)
#dzs_soil  = loaded_data['dzs_soil'].flatten()
#dzs_geo   = loaded_data['dzs_geo'].flatten()
#m2_coords = loaded_data['m2_coords']
loaded_gdf_dict = loaded_data['gdf_data']
gdf_reloaded = pd.DataFrame({
    'lon': loaded_gdf_dict['lon'][0, 0].flatten(),
    'lat': loaded_gdf_dict['lat'][0, 0].flatten(),
    'h_distance': loaded_gdf_dict['h_distance'][0, 0].flatten(),
    'elevation': loaded_gdf_dict['elevation'][0, 0].flatten()
})
geometry = [Point(xy) for xy in zip(gdf_reloaded['lon'], gdf_reloaded['lat'])]
hillslope_gdf = gpd.GeoDataFrame(gdf_reloaded, geometry=geometry)

# create hillslope polygon and shape object
xsec_plg = Polygon([hillslope_gdf.geometry[i] for i in range(hillslope_gdf.shape[0])])
xsec_plg_dict = {"type": "Feature", "id":0, "properties":{}, "geometry": mapping(xsec_plg)}
xsec_plg_dict_shply = watershed_workflow.utils.create_shply(xsec_plg_dict)

# convert to latlon crs, used in some plots
reproj_xsec_plg = watershed_workflow.warp.shape(xsec_plg_dict, crs_daymet, crs_latlon)
reproj_xsec_plg_shply = watershed_workflow.utils.create_shply(reproj_xsec_plg)

In [ ]:
hillslope_gdf

# Get MODIS-LAI for the watershed

MODIS product:

|Product|File name|Spatial resolution|Temporal resolution|Period|
|---|---|---|---|---|
|LAI|MCD15A3H.0.61_500m_aid0001.nc|500-m|4-day|2002-07-01 - present|
|Landcover|MCD12Q1.0.61_500m_aid0001.nc|500-m|yearly|2001-01-01 - present|
|ET|MOD16A2GF.0.61_500m_aid0001.nc|500-m|8-day|2000-01-01 - present|
|Snowcover|MOD10A2.0.61_500m_aid0001.nc|500-m|6-day?|2000-02-24 - present|

## MODIS-LAI data: load, subset, and plot

In [ ]:
#watershed_name = 'OakCreek'
data_raw_dir = f'./MODIS_raw/{watershed_name}/'
#data_processed_dir = f'../../data-processed/{name}/'

fname_lai = data_raw_dir + '/MCD15A3H.061_500m_aid0001.nc'
fname_lulc = data_raw_dir + '/MCD12Q1.061_500m_aid0001.nc'
fname_et = data_raw_dir + '/MOD16A2GF.061_500m_aid0001.nc'
fname_snowcover = data_raw_dir + '/MOD10A2.061_500m_aid0001.nc'

In [ ]:
dset = xr.open_dataset(fname_lai)
dset

In [ ]:
data = dset.Lai_500m
data.shape

In [ ]:
np.nanmin(data.values), np.nanmax(data.values)

In [ ]:
np.nanmean(data.values), np.nanmedian(data.values)

In [ ]:
plt.hist(data.values.flatten()[data.values.flatten()<1000], 100)
plt.show()

In [ ]:
mask_data = data.where((data < 6.5) & (data >= 0)) # try different threshold
mask_data.min(), mask_data.max()

In [ ]:
#mask_data.isel(time=-1).plot()

In [ ]:
mask_data.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
mask_data.rio.write_crs("epsg:4326", inplace=True)
clipped_data = mask_data.rio.clip(watershed_shape.geometry.apply(mapping), watershed_shape.crs, drop = True)

In [ ]:
LAI_data = clipped_data
LAI_data.shape

In [ ]:
LAI_data.isel(time=slice(1700, 1790, 8)).plot(x="lon", y='lat', col="time", col_wrap=4, robust=True, levels=np.linspace(0,5,51), cmap='Spectral_r')

In [ ]:
# Schneider Springs Fire: 2021-8-4 evening to ??

make_many_figs_and_gif = False
make_single_fig = True

# if make_many_figs_and_gif:
    
#     desktop_dir = os.environ.get('DESKTOPDIR')
#     temp_imgs_dir = desktop_dir + '/LAI_temp_imgs/'

#     try:
#         shutil.rmtree(temp_img_dir)
#     except OSError:
#         pass
#     os.mkdir(temp_img_dir)
    
#     for i in np.arange(int(LAI_data.shape[0]/10)):
#         fig, ax = plt.subplots(1, 1, figsize=(8,4))
#         LAI_data.isel(time=i).plot(ax=ax, levels=np.linspace(0,5,51), cmap='Spectral_r', extend='max')
#         # watershed_shape.boundary.plot(ax=ax, color='r')
#         ax.set_aspect('equal')
#         plt.tight_layout()
#         plt.savefig(temp_img_dir + str(i).zfill(4)+'.jpg')
#         plt.close()
#         if i % 100 == 0:
#             print(i, end=' ')
#     import imageio
#     from pygifsicle import optimize
#     images, image_file_names = [], []
#     for file_name in os.listdir(temp_img_dir):
#         if file_name.endswith('.jpg'):
#             image_file_names.append(file_name)       
#     # sorted_files = sorted(image_file_names, key=lambda y: int(y.split('_')[1]))
#     for i in range(len(image_file_names)):       
#         filetemp_img_dir = os.path.join(temp_img_dir, image_file_names[i])
#         images.append(imageio.imread(filetemp_img_dir))
#     imageio.mimsave(desktop_dir + f'{name}_LAI.gif', images, 'GIF', loop=1, fps=30)
#     optimize(desktop_dir + f'{name}_LAI.gif')
    
if make_single_fig:
    fig, axs = plt.subplots(1, 2, figsize=(10,4))
    ax1, ax2 = axs[0], axs[1]
    LAI_data.isel(time=1751).plot(ax=ax1, levels=np.linspace(0,5,51), cmap='Spectral_r', extend='max')
    LAI_data.isel(time=1763).plot(ax=ax2, levels=np.linspace(0,5,51), cmap='Spectral_r', extend='max')
    plt.tight_layout()
    plt.show()


In [ ]:
# Schneider Springs Fire: 2021-8-4 evening to ??
print("pre-fire time: " + LAI_data.time[1751].dt.strftime('%Y-%m-%d').item())
print("post-fire time: " + LAI_data.time[1763].dt.strftime('%Y-%m-%d').item())

fig, ax = plt.subplots(1, 1, figsize=(5,4))
dif = LAI_data.isel(time=1751) - LAI_data.isel(time=1763)
dif.plot(ax=ax, levels=np.linspace(-2.5,2.5,51), cmap='coolwarm', extend='both')
plt.title('positive: pre-fire > post-fire\nnegative: pre-fire < post-fire')
plt.tight_layout()
plt.show()

## MODIS-LULC data: load, subset, and plot

- LULC Land Use and Land Cover

In [ ]:
dset = xr.open_dataset(fname_lulc)
dset

In [ ]:
# counting LULC types
data = dset.LC_Type1
#ids, counts = np.unique(data.values[~np.isnan(data.values)], return_counts=True)

mask_data = data
mask_data.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
mask_data.rio.write_crs("epsg:4326", inplace=True)
landcover_data_modis_masked = mask_data.rio.clip(watershed_shape.geometry.apply(mapping), watershed_shape.crs, drop = True)

In [ ]:
# MODIS LULC labels
# Colors are based on NLCD LULC colors
lc_type1_colors = {
        -1:  ('Unclassified', (0.00000000000,  0.00000000000,  0.00000000000)),
        0: ('Open Water', (0.27843137255,  0.41960784314,  0.62745098039)),
        1: ('Evergreen Needleleaf Forests', (0.10980392157,  0.38823529412,  0.18823529412)),
        2: ('Evergreen Broadleaf Forests', (0.10980392157,  0.38823529412,  0.18823529412)),
        3: ('Deciduous Needleleaf Forests', (0.40784313726,  0.66666666667,  0.38823529412)),
        4: ('Deciduous Broadleaf Forests', (0.40784313726,  0.66666666667,  0.38823529412)),
        5: ('Mixed Forests', (0.70980392157,  0.78823529412,  0.55686274510)),
        6: ('Closed Shrublands', (0.80000000000,  0.72941176471,  0.48627450980)),
        7: ('Open Shrublands', (0.80000000000,  0.72941176471,  0.48627450980)),
        8: ('Woody Savannas', (0.60980392157,  0.68823529412,  0.55686274510)),
        9: ('Savannas', (0.70980392157,  0.78823529412,  0.55686274510)),
        10: ('Grasslands', (0.88627450980,  0.88627450980,  0.75686274510)),
        11: ('Permanent Wetlands', (0.43921568628,  0.63921568628,  0.72941176471)),
        12: ('Croplands', (0.66666666667,  0.43921568628,  0.15686274510)),
        13: ('Urban and Built up lands', (0.86666666667,  0.78823529412,  0.78823529412)),
        14: ('Cropland Natural Vegetation Mosaics', (0.66666666667,  0.43921568628,  0.15686274510)),
        15: ('Permanent Snow and Ice', (0.81960784314,  0.86666666667,  0.97647058824)),
        16: ('Barren Land', (0.69803921569,  0.67843137255,  0.63921568628)),
        17: ('Water Bodies', (0.27843137255,  0.41960784314,  0.62745098039)),
    } 

In [ ]:
lc_colors = lc_type1_colors

In [ ]:
watershed_ids, watershed_counts = np.unique(landcover_data_modis_masked.values[~np.isnan(landcover_data_modis_masked.values)], return_counts=True)

watershed_colors = [lc_colors[i][1] for i in watershed_ids]
watershed_labels = [lc_colors[i][0] for i in watershed_ids]

watershed_ids, watershed_counts, watershed_colors, watershed_labels

In [ ]:
watershed_ids.max()

In [ ]:
labelsp1 = [lc_colors[i][0] for i in lc_colors]

counts, bins = np.histogram(landcover_data_modis_masked, range=[-1,17], bins=18)
plt.hist(bins[:-1], bins=18, range=[-1,17], weights=counts)
plt.xlabel("MODIS LULC")
plt.ylabel("Counts")
plt.title("Histogram of number of pixels per LULC types")
plt.xticks(bins+0.5,labelsp1,rotation=90)
plt.show()

In [ ]:
# LULC data plot - spatial distribution at a time slice

# [Yi] add an id to ensure plot(levels=watershed_ids, colors = watershed_colors, ax=ax, add_colorbar = False) display correctly
# watershed_ids_ext4plot = [0] + watershed_ids.tolist() + [watershed_ids.max()+1]
# watershed_colors_ext4plot = ['grey'] + watershed_colors + ['k']
# watershed_labels_ext4plot = ['None'] + watershed_labels + ['']
watershed_ids_ext4plot = watershed_ids.tolist() + [watershed_ids.max()+1]
watershed_colors_ext4plot = watershed_colors + ['k']
watershed_labels_ext4plot = watershed_labels + ['']

fig, ax = plt.subplots(1,1, figsize=(8,4))
g = landcover_data_modis_masked.isel(time = -1).plot(levels=watershed_ids_ext4plot, 
                                                     colors = watershed_colors_ext4plot, ax=ax, add_colorbar = False)

midpoints = 0.5 * (np.array(watershed_ids_ext4plot[:-1]) + np.array(watershed_ids_ext4plot[1:]))
cb = plt.colorbar(g)
cb.set_ticks(midpoints)
cb.ax.tick_params(size=0)
cb.set_ticklabels(watershed_labels_ext4plot[:-1])

watershed_shape.boundary.plot(ax=ax, color ='r')
ax.set_aspect('equal')

## Merge LAI and LULC to a dataframe

In [ ]:
# LULC data plot - temporal plot

ilandcover = landcover_data_modis_masked.sel(time="2023-01-01").values[0,:,:] # in watershed-workflow, by default it's using the latest year LULC.
print(ilandcover.shape)

# lc_ids = np.unique(landcover_data_modis_masked.values[~np.isnan(landcover_data_modis_masked.values)])
# print(lc_ids)
# lc_labels = [lc_colors[i][0] for i in lc_ids]
# print(lc_labels)
times = LAI_data.time.values
print(times)
print(len(times))

In [ ]:
# calculate mean LAI for each landcover type
LAI_lc = []

for itime in times:
#itime = times[0]
    iCC_LAI = LAI_data.sel(time=itime).values
    iLAI_lc = []
    for i,ilabel in zip(watershed_ids, watershed_labels):
        idx = np.where(ilandcover == i)
        coords = list(zip(idx[0], idx[1]))
        # choose mean of the LAI for each landcover type
        iLAI = np.nanmean(np.array([iCC_LAI[i] for i in coords]))
        iLAI_lc.append(iLAI)
    LAI_lc.append(iLAI_lc)
    
print(len(LAI_lc))
print(LAI_lc[0])

In [ ]:
# generate dataframe: LAI of each LC with time
LAI_df = pd.DataFrame(LAI_lc, columns=watershed_labels)
LAI_df['datetime'] = LAI_data.indexes['time'].to_datetimeindex()
# LAI_df = LAI_df.iloc[21:-29]
LAI_df = LAI_df.iloc[0:]
LAI_df.iloc[0,-1] = LAI_df.iloc[0,-1]+(LAI_df.iloc[1,-1]-LAI_df.iloc[0,-1])/4
LAI_df.set_index('datetime', inplace=True)
LAI_df['time [s]'] = (LAI_df.index - LAI_df.index[0]).total_seconds()
LAI_df

In [ ]:
LAI_df.plot()
plt.ylabel('LAI [-]')
plt.xlim(["2005-01-01", "2010-01-01"])
plt.ylim(-0.5,5)

### [optional] LAI varition plot

In [ ]:
# if want to see the spatial variation

# Dictionary to store spatial variation (per coordinate) for each watershed
spatial_variation = {label: {} for label in watershed_labels}

for itime in times:
    iCC_LAI = LAI_data.sel(time=itime).values
    for i, ilabel in zip(watershed_ids, watershed_labels):
        # Get all coordinates for the current landcover type
        idx = np.where(ilandcover == i)
        coords = list(zip(idx[0], idx[1]))
        
        # Iterate over each coordinate and store time series
        for coord in coords:
            if coord not in spatial_variation[ilabel]:
                spatial_variation[ilabel][coord] = []
            spatial_variation[ilabel][coord].append(iCC_LAI[coord])

# Plot spatial variations for each landcover type
times_datetime = LAI_data.indexes['time'].to_datetimeindex()

# for ilabel, coord_data in spatial_variation.items():
#     plt.figure(figsize=(10, 6))
#     for coord, values in coord_data.items():
#         plt.plot(times_datetime, values, label=f"Coord: {coord}", alpha=0.5, linewidth=0.8)
    
#     plt.title(f"Spatial Variation of LAI for {ilabel}")
#     plt.xlabel("Time")
#     plt.ylabel("LAI [-]")
#     plt.ylim(-0.5, 5)
#     #plt.legend(fontsize=8, loc="upper right", framealpha=0.7, ncol=2)
#     plt.tight_layout()
#     plt.show()

import seaborn as sns
color_palette = sns.color_palette("tab10", len(watershed_labels))
landcover_colors = {label: color for label, color in zip(watershed_labels, color_palette)}

for ilabel, coord_data in spatial_variation.items():
    plt.figure(figsize=(12, 6))

    # Convert the time series data into a DataFrame for easier calculation of stats
    coord_df = pd.DataFrame.from_dict(coord_data, orient='columns')  # Columns are coordinates
    coord_df.index = times_datetime

    # Calculate metrics
    mean_series = coord_df.mean(axis=1)  # Mean across coordinates
    p10_series = coord_df.quantile(0.1, axis=1)  # 10th percentile
    p90_series = coord_df.quantile(0.9, axis=1)  # 90th percentile

    # Plot the mean line
    plt.plot(
        times_datetime, 
        mean_series, 
        color=landcover_colors[ilabel], 
        label=f"{ilabel} Mean", 
        linewidth=2
    )

    # Plot the shaded area for the 10th to 90th percentile range
    plt.fill_between(
        times_datetime, 
        p10_series, 
        p90_series, 
        color=landcover_colors[ilabel], 
        alpha=0.3,  # Transparency for the shaded area
        label=f"{ilabel} 10th-90th Percentile"
    )

    # Add labels, legends, and formatting
    plt.title(f"LAI for {ilabel}: Mean and 10th-90th Percentile")
    plt.xlabel("Time")
    plt.ylabel("LAI [-]")
    plt.ylim(-0.5, 5)
    #plt.legend(fontsize=10)
    plt.grid(alpha=0.5)
    plt.tight_layout()
    plt.show()

## Process LAI_df further; crosswalk with NLCD

### Map MODIS landcover to NLCD landcover
We use NLCD for its high resolution, but need LAI from MODIS. Therefore we will get NLCD on the domain, and form the "crosswalk" between NLCD and MODIS.

[note]: 
- code largely borrowed from get_MODIS_LAI.ipynb in watershed-workflow
- to-do 1, Zhi did threshold cutting for LULC types with < 5% of the pixel coverage
- to-do 2, read LAI and LULC to standard FileManager instance in above part

In [ ]:
# get the NLCD data on the polygon
sources = watershed_workflow.source_list.get_default_sources() # only use the sources['land cover']

from shapely.ops import unary_union
watershed_polygon = unary_union(watershed_shape.geometry)  # Combine possible multi polygons into one; equavalent to watershed.exterior()

nlcd_profile, nlcd_raster = watershed_workflow.get_raster_on_shape(sources['land cover'], watershed_polygon, crs_daymet)
# nlcd = watershed_workflow.values_from_raster(surface_centroid, crs, lc_raster, lc_profile)

In [ ]:
# plot NLCD and MODIS land use
fig_nlcd, ax_nlcd = watershed_workflow.plot.get_ax(nlcd_profile['crs'], nrow=1, ncol=1, index=1, figsize=(15,5))

# plot the NLCD image
# -- get the NLCD colormap which uses official NLCD colors and labels
nlcd_indices, nlcd_cmap, nlcd_norm, nlcd_ticks, nlcd_labels = \
            watershed_workflow.colors.generate_nlcd_colormap(np.unique(nlcd_raster))

watershed_workflow.plot.raster(nlcd_profile, nlcd_raster, ax=ax_nlcd, cmap=nlcd_cmap, norm=nlcd_norm)
watershed_workflow.plot.shply(watershed_workflow.warp.shply(watershed_polygon, crs_daymet, nlcd_profile['crs']), nlcd_profile['crs'], 'k', ax_nlcd)
watershed_workflow.colors.colorbar_index(ncolors=len(np.unique(nlcd_raster)), cmap=nlcd_cmap, labels=nlcd_labels, ax=ax_nlcd) 
ax_nlcd.set_title("NLCD Index")

# plot the MODIS landcover
## read the MODIS with filenames
from watershed_workflow.sources.manager_modis_appeears import FileManagerMODISAppEEARS
fmodis = FileManagerMODISAppEEARS()
modislulc_data = fmodis.get_data(filenames=[fname_lulc], variables=["LULC"])

modis_raster = modislulc_data['LULC'].data[-1]
modis_profile = modislulc_data['LULC'].profile

fig_modis, ax_modis = watershed_workflow.plot.get_ax(modis_profile['crs'], nrow=1, ncol=1, index=1, figsize=(15,5))

modis_indices, modis_cmap, modis_norm, modis_ticks, modis_labels = \
            watershed_workflow.colors.generate_modis_colormap(np.unique(modis_raster))
print(modis_indices, modis_labels)
print(modis_cmap(8))
print(modis_cmap(4))
watershed_workflow.plot.raster(modis_profile, modis_raster, ax=ax_modis, cmap=modis_cmap, norm=modis_norm)
watershed_workflow.plot.shply(watershed_workflow.warp.shply(watershed_polygon, crs_daymet, modis_profile['crs']), modis_profile['crs'], 'k', ax_modis)
watershed_workflow.colors.colorbar_index(ncolors=len(np.unique(modis_raster)), cmap=modis_cmap, labels=modis_labels, ax=ax_modis) 
ax_modis.set_title("MODIS LULC Index")

In [ ]:
# form the crosswalk and plot a correlation matrix
crosswalk = watershed_workflow.land_cover_properties.computeCrosswalkCorrelation(modislulc_data['LULC'].profile, 
                                                                                 modislulc_data['LULC'].data[-1],
                                                                                 nlcd_profile, nlcd_raster)

for key, value in crosswalk.items():
    print(f"{key}: {value}")

### Select dominant LULC types and use their LAI values for ATS run

In [ ]:
cut_threshold = 0.01

In [ ]:
lulc_ids, lulc_counts = np.unique(landcover_data_modis_masked.values[~np.isnan(landcover_data_modis_masked.values)], return_counts=True)

sum1 = sum(counts)
cutoff = sum1*cut_threshold #screen LULC types with 5% of the pixel coverage cutoff

filtered_lulc_ids = np.delete(lulc_ids, np.argwhere(lulc_counts < int(cutoff)))
print("LULC ids pass cutoff: " + str(filtered_lulc_ids))

filtered_lulc_colors = [lc_colors[i][1] for i in filtered_lulc_ids]
filtered_lulc_labels = [lc_colors[i][0] for i in filtered_lulc_ids]

print("their labels1: " + str(filtered_lulc_labels))

In [ ]:
# dominant LULC
countLULCclass = len(filtered_lulc_ids)

if(countLULCclass >= 5):
    dom = np.argpartition(-lulc_counts, range(5))[:5]
    print(lulc_ids[dom])  # prints the 5 most frequent LULC IDs
else:
    dom = np.argpartition(-lulc_counts, range(countLULCclass))[:countLULCclass]
    print(lulc_ids[dom])  # prints the most frequent LULC ID

In [ ]:
# dominant LULC and label
if(countLULCclass >= 5):
    a = lulc_ids[dom]
    LULC1 = a[0]
    LULC2 = a[1]
    LULC3 = a[2]
    LULC4 = a[3]
    LULC5 = a[4]
    LULC1label = lc_colors[LULC1][0]
    LULC2label = lc_colors[LULC2][0]
    LULC3label = lc_colors[LULC3][0]
    LULC4label = lc_colors[LULC4][0]
    LULC5label = lc_colors[LULC5][0]
elif(countLULCclass == 4):
    a = lulc_ids[dom]
    LULC1 = a[0]
    LULC2 = a[1]
    LULC3 = a[2]
    LULC4 = a[3]
    LULC1label = lc_colors[LULC1][0]
    LULC2label = lc_colors[LULC2][0]
    LULC3label = lc_colors[LULC3][0]
    LULC4label = lc_colors[LULC4][0]
elif(countLULCclass == 3):
    a = lulc_ids[dom]
    LULC1 = a[0]
    LULC2 = a[1]
    LULC3 = a[2]
    LULC1label = lc_colors[LULC1][0]
    LULC2label = lc_colors[LULC2][0]
    LULC3label = lc_colors[LULC3][0]
elif(countLULCclass == 2):
    a = lulc_ids[dom]
    LULC1 = a[0]
    LULC2 = a[1]
    LULC1label = lc_colors[LULC1][0]
    LULC2label = lc_colors[LULC2][0]
else:
    a = lulc_ids[dom]
    LULC1 = a[0]
    LULC1label = lc_colors[LULC1][0]

### MODIS and NLCD crosswalk

<font color='green'> Users may change the crosswalk between MODIS and NLCD labels based on their study area characterisctics

In [ ]:
#Colors are based on NLCD LULC colors
MODIS_labels = ['Unclassified', 
                'Open Water', 
                'Evergreen Needleleaf Forests', 
                'Evergreen Broadleaf Forests',
                'Deciduous Needleleaf Forests', 
                'Deciduous Broadleaf Forests', 
                'Mixed Forests', 
                'Closed Shrublands', 
                'Open Shrublands', 
                'Woody Savannas', 
                'Savannas', 
                'Grasslands', 
                'Permanent Wetlands', 
                'Croplands', 
                'Urban and Built up lands', 
                'Cropland Natural Vegetation Mosaics', 
                'Permanent Snow and Ice', 
                'Barren Land', 
                'Water Bodies']

In [ ]:
NLCD_labels = ['None',
               'Open Water',
               'Evergreen Forest',
               'Evergreen Forest',
               'Deciduous Forest',
               'Deciduous Forest',
               'Mixed Forest',
               'Shrub/Scrub',
               'Shrub/Scrub',
               'Woody Wetlands',
               'Pasture/Hay',
               'Grassland/Herbaceous',
               'Emergent Herbaceous Wetlands',
               'Cultivated Crops',
               'Developed, Medium Intensity',
               'Cultivated Crops',
               'Perrenial Ice/Snow',
               'Barren Land',
               'Open Water']

NLCD_labels = [_lb.replace('/',' ') for _lb in NLCD_labels]
print(NLCD_labels) 

In [ ]:
compare_labels = pd.DataFrame({'MODIS_labels': MODIS_labels, 'NLCD_labels': NLCD_labels})
compare_labels

In [ ]:
LULC1label, LULC2label, LULC3label, LULC4label

In [ ]:
# 1st step: change MODIS labels to NLCD labels directly

nlcd_LAI = LAI_df.copy()

if(countLULCclass >= 5):
    NLCDLULC1label=compare_labels[compare_labels['MODIS_labels']==LULC1label]
    NLCDLULC1label1 = NLCDLULC1label['NLCD_labels'].values[0]
    NLCDLULC2label=compare_labels[compare_labels['MODIS_labels']==LULC2label]
    NLCDLULC2label1 = NLCDLULC2label['NLCD_labels'].values[0]
    NLCDLULC3label=compare_labels[compare_labels['MODIS_labels']==LULC3label]
    NLCDLULC3label1 = NLCDLULC3label['NLCD_labels'].values[0]
    NLCDLULC4label=compare_labels[compare_labels['MODIS_labels']==LULC4label]
    NLCDLULC4label1 = NLCDLULC4label['NLCD_labels'].values[0]
    NLCDLULC5label=compare_labels[compare_labels['MODIS_labels']==LULC5label]
    NLCDLULC5label1 = NLCDLULC5label['NLCD_labels'].values[0]
    
    nlcd_LAI[f'NLCD {NLCDLULC1label1} LAI [-]'] = LAI_df[LULC1label]
    nlcd_LAI[f'NLCD {NLCDLULC2label1} LAI [-]'] = LAI_df[LULC2label]
    nlcd_LAI[f'NLCD {NLCDLULC3label1} LAI [-]'] = LAI_df[LULC3label]
    nlcd_LAI[f'NLCD {NLCDLULC4label1} LAI [-]'] = LAI_df[LULC4label]
    nlcd_LAI[f'NLCD {NLCDLULC5label1} LAI [-]'] = LAI_df[LULC5label]
    
    nlcd_LAI.plot(y= [f'NLCD {NLCDLULC1label1} LAI [-]', f'NLCD {NLCDLULC2label1} LAI [-]', f'NLCD {NLCDLULC3label1} LAI [-]', f'NLCD {NLCDLULC4label1} LAI [-]', f'NLCD {NLCDLULC5label1} LAI [-]'], lw = 1)
    plt.ylabel("LAI [-]")
    plt.xlim(datetime(2002,10,1), datetime(2025,1,25))

elif(countLULCclass == 4):
    NLCDLULC1label=compare_labels[compare_labels['MODIS_labels']==LULC1label]
    NLCDLULC1label1 = NLCDLULC1label['NLCD_labels'].values[0]
    NLCDLULC2label=compare_labels[compare_labels['MODIS_labels']==LULC2label]
    NLCDLULC2label1 = NLCDLULC2label['NLCD_labels'].values[0]
    NLCDLULC3label=compare_labels[compare_labels['MODIS_labels']==LULC3label]
    NLCDLULC3label1 = NLCDLULC3label['NLCD_labels'].values[0]
    NLCDLULC4label=compare_labels[compare_labels['MODIS_labels']==LULC4label]
    NLCDLULC4label1 = NLCDLULC4label['NLCD_labels'].values[0]
    
    nlcd_LAI[f'NLCD {NLCDLULC1label1} LAI [-]'] = LAI_df[LULC1label]
    nlcd_LAI[f'NLCD {NLCDLULC2label1} LAI [-]'] = LAI_df[LULC2label]
    nlcd_LAI[f'NLCD {NLCDLULC3label1} LAI [-]'] = LAI_df[LULC3label]
    nlcd_LAI[f'NLCD {NLCDLULC4label1} LAI [-]'] = LAI_df[LULC4label]
    
    nlcd_LAI.plot(y= [f'NLCD {NLCDLULC1label1} LAI [-]', f'NLCD {NLCDLULC2label1} LAI [-]', f'NLCD {NLCDLULC3label1} LAI [-]', f'NLCD {NLCDLULC4label1} LAI [-]'], lw = 1)
    plt.ylabel("LAI [-]")
    plt.xlim(datetime(2002,10,1), datetime(2025,1,25))

elif(countLULCclass == 3):
    NLCDLULC1label=compare_labels[compare_labels['MODIS_labels']==LULC1label]
    NLCDLULC1label1 = NLCDLULC1label['NLCD_labels'].values[0]
    NLCDLULC2label=compare_labels[compare_labels['MODIS_labels']==LULC2label]
    NLCDLULC2label1 = NLCDLULC2label['NLCD_labels'].values[0]
    NLCDLULC3label=compare_labels[compare_labels['MODIS_labels']==LULC3label]
    NLCDLULC3label1 = NLCDLULC3label['NLCD_labels'].values[0]
    
    nlcd_LAI[f'NLCD {NLCDLULC1label1} LAI [-]'] = LAI_df[LULC1label]
    nlcd_LAI[f'NLCD {NLCDLULC2label1} LAI [-]'] = LAI_df[LULC2label]
    nlcd_LAI[f'NLCD {NLCDLULC3label1} LAI [-]'] = LAI_df[LULC3label]
    
    nlcd_LAI.plot(y= [f'NLCD {NLCDLULC1label1} LAI [-]', f'NLCD {NLCDLULC2label1} LAI [-]', f'NLCD {NLCDLULC3label1} LAI [-]'], lw = 1)
    plt.ylabel("LAI [-]")
    plt.xlim(datetime(2002,10,1), datetime(2025,1,25))
    
elif(countLULCclass == 2):
    NLCDLULC1label=compare_labels[compare_labels['MODIS_labels']==LULC1label]
    NLCDLULC1label1 = NLCDLULC1label['NLCD_labels'].values[0]
    NLCDLULC2label=compare_labels[compare_labels['MODIS_labels']==LULC2label]
    NLCDLULC2label1 = NLCDLULC2label['NLCD_labels'].values[0]
    
    nlcd_LAI[f'NLCD {NLCDLULC1label1} LAI [-]'] = LAI_df[LULC1label]
    nlcd_LAI[f'NLCD {NLCDLULC2label1} LAI [-]'] = LAI_df[LULC2label]
    
    nlcd_LAI.plot(y= [f'NLCD {NLCDLULC1label1} LAI [-]', f'NLCD {NLCDLULC2label1} LAI [-]'], lw = 1)
    plt.ylabel("LAI [-]")
    plt.xlim(datetime(2002,10,1), datetime(2025,1,25))
    
else:
    NLCDLULC1label=compare_labels[compare_labels['MODIS_labels']==LULC1label]
    NLCDLULC1label1 = NLCDLULC1label['NLCD_labels'].values[0]
    
    nlcd_LAI[f'NLCD {NLCDLULC1label1} LAI [-]'] = LAI_df[LULC1label]
    
    nlcd_LAI.plot(y= [f'NLCD {NLCDLULC1label1} LAI [-]'], lw = 1)
    plt.ylabel("LAI [-]")
    plt.xlim(datetime(2002,10,1), datetime(2025,1,25))

In [ ]:
LULC1label, LULC2label, LULC3label, LULC4label

In [ ]:
NLCDLULC1label1, NLCDLULC2label1, NLCDLULC3label1, NLCDLULC4label1

In [ ]:
# 2nd step: adjust NLCD labels above based on NLCD map in full_workflow.ipynb
# for Oak Creek here,
# - keep Grassland Herbaceous and Evergreen Forest
# - average NLCDLULC2+NLCDLULC3, i.e. Woody Wetlands and Pasture Hay -> Shrub Scrub

for i in [f'NLCD {NLCDLULC1label1} LAI [-]', f'NLCD {NLCDLULC2label1} LAI [-]', f'NLCD {NLCDLULC3label1} LAI [-]', f'NLCD {NLCDLULC4label1} LAI [-]']:
    if i == f'NLCD {NLCDLULC2label1} LAI [-]':
        a = nlcd_LAI[i].values
        print(a)
    if i == f'NLCD {NLCDLULC3label1} LAI [-]':
        b = nlcd_LAI[i].values
        print(b)
c = (a+b)/2
print(c)

### Save processed LAI with NLCD labels to hdf5

In [ ]:
outputs['modis_filename_watershed_raw'] = f'../data-processed/{watershed_name}/{watershed_name}_MODIS_LAI_20021001_20250125.h5'

with h5.File(outputs['modis_filename_watershed_raw'], 'w') as fout:
    for i in ['time [s]', f'NLCD {NLCDLULC1label1} LAI [-]', f'NLCD {NLCDLULC2label1} LAI [-]', f'NLCD {NLCDLULC3label1} LAI [-]', f'NLCD {NLCDLULC4label1} LAI [-]']:
        if i == f'NLCD {NLCDLULC2label1} LAI [-]':
            fout.create_dataset('NLCD Shrub Scrub LAI [-]', data=c)
        elif i == f'NLCD {NLCDLULC3label1} LAI [-]':
            continue
        else:
            fout.create_dataset(i, data=nlcd_LAI[i].values)

## Process LAI data with NLCD labels for ATS

- interpolate every 4d to daily
- remove leap year (same as DayMet, remove 12/31 if it's a leap year)
- smooth
- typical year

### interpolate, remove leap year, and smooth

In [ ]:
# use data from start_year to end_year, to generate typical year data. 
# Then cyclic for nyears_cyclic_steadystate years.
#start_year = 2013 # in config.json now
#end_year = 2023
#nyears_cyclic_steadystate = 10

In [ ]:
d = h5.File(outputs['modis_filename_watershed_raw'],'r')
df = pd.DataFrame()
for k in d.keys():
    df[k] = d[k][:]
df['time [d]'] = df['time [s]']/86400

df

In [ ]:
# interpolate this time series into a daily time series
# ts = np.arange(8214, 14600, 1)
ts = np.arange(df['time [d]'].values[-1]+1)
df_interp = pd.DataFrame()
df_interp['time [d]'] = ts

for k in df.keys():
    if k != 'time [s]':
        f = scipy.interpolate.interp1d(df['time [d]'][:], df[k][:])
        df_interp[k] = f(ts)

df = df_interp
df['datetime'] = pd.to_datetime(df['time [d]'], unit='D', origin=pd.Timestamp('2002-10-01'))

df

In [ ]:
# v20250423
# crop MODIS-LAI data based on startdate and enddate
startdate = f"{start_year}-1-1"
enddate = f"{end_year+1}-1-1"

mask = (df['datetime'] >= startdate) & (df['datetime'] <= enddate)
df_cropped = df[mask]
# Reset the index of the cropped DataFrame
df_cropped = df_cropped.reset_index(drop=True)
# Calculate the time difference from the new start date
start_datetime = pd.to_datetime(startdate)
df_cropped['time [d]'] = (df_cropped['datetime'] - start_datetime).dt.days
df = df_cropped
df

In [ ]:
leap_index = []
for i in range(len(df)):
    if '02-29' in str(df.iloc[i,-1]):
        leap_index.append(i)
        print(i, df.iloc[i,-1])

In [ ]:
df.drop(leap_index, inplace=True)
df.reset_index(drop=True, inplace=True)
df['time [d]'] = np.arange(len(df))

In [ ]:
# verify any '02-29' left
for i in range(len(df)):
    if '02-29' in str(df.iloc[i,-1]):
        print(i, df.iloc[i,-1])

In [ ]:
df.drop(columns=['datetime'], inplace=True)
df

In [ ]:
# smooth the data
df_smooth = pd.DataFrame()
df_smooth['time [d]'] = df['time [d]']
for k in df.keys():
    if k != 'time [d]':
        df_smooth[k] = scipy.signal.savgol_filter(df[k], 101, 3)


df_smooth.iloc[:,1:].plot(figsize=(8,4))
# # plot comparison
# fig = plt.figure()
# axs = fig.subplots(3,1)
# # plot(df, '-', axs)
# plot(df_smooth, '-', axs)
# plt.tight_layout()
plt.show()
        
#df_smooth

### Save MODIS - write to disk

In [ ]:
# add time back and write to disk
outputs['modis_filename_watershed_smoothed'] = f'../data-processed/{watershed_name}/{watershed_name}_MODIS_LAI_{startdate}_{enddate}_smoothed.h5'

df_smooth['time [s]'] = df_smooth['time [d]']*86400
with h5.File(outputs['modis_filename_watershed_smoothed'],'w') as fid:
    for k in df_smooth:
        fid.create_dataset(k, data=df_smooth[k][:])

In [ ]:
# typical year

df = df_smooth
# split into n_years dataframes, one per year
df_yr = []
nyears = np.ceil((df["time [d]"].iloc[-1]+1)/365).astype(int)
for year in range(nyears):
    yr = df.loc[df_interp['time [d]'] >= year*365]
    df_yr.append(yr.loc[yr['time [d]'] < (year+1)*365])

# average across the years
df_avg = pd.DataFrame()
for yr in df_yr:
    for k in yr.keys():
        if not k.startswith('time'):
            if k in df_avg:
                df_avg[k] = df_avg[k].array + yr[k].array
            else:
                df_avg[k] = yr[k].copy()

for k in df_avg.keys():
    df_avg[k] = df_avg[k][:] / len(df_yr)

df_avg['time [d]'] = df['time [d]']

df_avg.iloc[:,:-1].plot(figsize=(4,4))
# fig = plt.figure()
# axs = fig.subplots(3,1)
# plot(df_avg, '-', axs)
# plt.tight_layout()
plt.show()

df_avg

In [ ]:
#nyears = end_year-start_year+1 # to match DayMet 1980-2022
nyears = nyears_cyclic_steadystate # to match typical DayMet for cyclic spin-up

# replicate nyears times to make nyears years (remem)
# tile all data to repeat n_year times
df_repeat = pd.DataFrame()
for key in df_avg:
    if not key.startswith('time'):
        df_repeat[key] = np.tile(df_avg[key].array, nyears)
        assert(len(df_repeat) == nyears*365)

# time is simply daily data
df_repeat['time [d]'] = np.arange(0., nyears * 365., 1.)
df_repeat['time [s]'] = 86400*df_repeat['time [d]']

# plot
df_repeat.iloc[:,:-2].plot(figsize=(8,4))
# plot this and make sure it looks right
# fig = plt.figure()
# axs = fig.subplots(3,1)
# plot(df_repeat, '-', axs)
# plt.tight_layout()
plt.show()

df_repeat

In [ ]:
# write to disk
outputs['modis_typical_filename_watershed'] = f'../data-processed/{watershed_name}/{watershed_name}_MODIS_LAI_typical{nyears}yr_{start_year}_{end_year}.h5'

if 'time [d]' in df_repeat.columns:
    df_repeat = df_repeat.drop(columns=['time [d]'])
with h5.File(outputs['modis_typical_filename_watershed'],'w') as fid:
    for k in df_repeat:
        fid.create_dataset(k, data=df_repeat[k][:])

# Get MODIS-LAI for the hillslope site

## Plot NLCD and MODIS-LULC with 2D transect

In [ ]:
m2_mat_filename =  f'../data-processed/{site_name}/startendcoords_{site_name}.mat'
loaded_data  = loadmat(m2_mat_filename)
start_coords = loaded_data['start_coords'].flatten()
end_coords   = loaded_data['end_coords'].flatten()

print(start_coords)
print(end_coords)

In [ ]:
# Bounding box in crs_daymet
dx = 5000/2
dy = 4000/2
xmin = (start_coords[0]+end_coords[0])/2 - dx/2
xmax = (start_coords[0]+end_coords[0])/2 + dx/2
ymin = (start_coords[1]+end_coords[1])/2 - dy/2
ymax = (start_coords[1]+end_coords[1])/2 + dy/2
print("Bounding box in crs_daymet: "+ str([xmin, xmax, ymin, ymax]))

# Determine Bounding box in nlcd_profile['crs']
xmin_nlcd, ymin_nlcd = pyproj.transform(crs_daymet, nlcd_profile['crs'], xmin, ymin)
xmax_nlcd, ymax_nlcd = pyproj.transform(crs_daymet, nlcd_profile['crs'], xmax, ymax)
print("Bounding box in nlcd_profile['crs']: "+ str([xmin_nlcd, xmax_nlcd, ymin_nlcd, ymax_nlcd]))

# Determine Bounding box in modis_profile['crs']
xmin_modis, ymin_modis = pyproj.transform(crs_daymet, modis_profile['crs'], xmin, ymin)
xmax_modis, ymax_modis = pyproj.transform(crs_daymet, modis_profile['crs'], xmax, ymax)
print("Bounding box in modis_profile['crs']: "+ str([xmin_modis, xmax_modis, ymin_modis, ymax_modis]))

In [ ]:
# plot NLCD
fig_nlcd, ax_nlcd = watershed_workflow.plot.get_ax(nlcd_profile['crs'], nrow=1, ncol=1, index=1, figsize=(15,5))
watershed_workflow.plot.raster(nlcd_profile, nlcd_raster, ax=ax_nlcd, cmap=nlcd_cmap, norm=nlcd_norm)
watershed_workflow.plot.shply(watershed_workflow.warp.shply(watershed_polygon, crs_daymet, nlcd_profile['crs']), nlcd_profile['crs'], 'k', ax_nlcd)
watershed_workflow.colors.colorbar_index(ncolors=len(np.unique(nlcd_raster)), cmap=nlcd_cmap, labels=nlcd_labels, ax=ax_nlcd) 
watershed_workflow.plot.shply(watershed_workflow.warp.shply(xsec_plg_dict_shply, crs_daymet, nlcd_profile['crs']), nlcd_profile['crs'], 'r', ax_nlcd)
ax_nlcd.set_title("NLCD Index")

In [ ]:
# plot NLCD - zoom in
fig_nlcd, ax_nlcd = watershed_workflow.plot.get_ax(nlcd_profile['crs'], nrow=1, ncol=1, index=1, figsize=(15,5))
watershed_workflow.plot.raster(nlcd_profile, nlcd_raster, ax=ax_nlcd, cmap=nlcd_cmap, norm=nlcd_norm)
watershed_workflow.plot.shply(watershed_workflow.warp.shply(watershed_polygon, crs_daymet, nlcd_profile['crs']), nlcd_profile['crs'], 'k', ax_nlcd)
watershed_workflow.colors.colorbar_index(ncolors=len(np.unique(nlcd_raster)), cmap=nlcd_cmap, labels=nlcd_labels, ax=ax_nlcd) 
watershed_workflow.plot.shply(watershed_workflow.warp.shply(xsec_plg_dict_shply, crs_daymet, nlcd_profile['crs']), nlcd_profile['crs'], 'r', ax_nlcd)
ax_nlcd.set_title("NLCD Index - zoom in")
ax_nlcd.set_xlim(xmin_nlcd, xmax_nlcd)
ax_nlcd.set_ylim(ymin_nlcd, ymax_nlcd)

In [ ]:
# plot MODIS landcover
fig_modis, ax_modis = watershed_workflow.plot.get_ax(modis_profile['crs'], nrow=1, ncol=1, index=1, figsize=(15,5))
watershed_workflow.plot.raster(modis_profile, modis_raster, ax=ax_modis, cmap=modis_cmap, norm=modis_norm)
watershed_workflow.plot.shply(watershed_workflow.warp.shply(watershed_polygon, crs_daymet, modis_profile['crs']), modis_profile['crs'], 'k', ax_modis)
watershed_workflow.colors.colorbar_index(ncolors=len(np.unique(modis_raster)), cmap=modis_cmap, labels=modis_labels, ax=ax_modis) 
watershed_workflow.plot.shply(watershed_workflow.warp.shply(xsec_plg_dict_shply, crs_daymet, modis_profile['crs']), modis_profile['crs'], 'r', ax_modis)
ax_modis.set_title("MODIS LULC Index")

In [ ]:
# plot MODIS landcover - zoom in
fig_modis, ax_modis = watershed_workflow.plot.get_ax(modis_profile['crs'], nrow=1, ncol=1, index=1, figsize=(15,5))
watershed_workflow.plot.raster(modis_profile, modis_raster, ax=ax_modis, cmap=modis_cmap, norm=modis_norm)
watershed_workflow.plot.shply(watershed_workflow.warp.shply(watershed_polygon, crs_daymet, modis_profile['crs']), modis_profile['crs'], 'k', ax_modis)
watershed_workflow.colors.colorbar_index(ncolors=len(np.unique(modis_raster)), cmap=modis_cmap, labels=modis_labels, ax=ax_modis) 
watershed_workflow.plot.shply(watershed_workflow.warp.shply(xsec_plg_dict_shply, crs_daymet, modis_profile['crs']), modis_profile['crs'], 'r', ax_modis)
ax_modis.set_title("MODIS LULC Index - zoom in")
ax_modis.set_xlim(ymin_modis, ymax_modis)
ax_modis.set_ylim(xmin_modis, xmax_modis)

In [ ]:
#[to do] a downscale method is needed to generate fine-scale LAI product for hillslope

In [ ]:
outputs